In [72]:
import os

from aeon.datasets import load_classification
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import polars as pl
import numpy as np

In [73]:
multivariate_equal_length = [
    "ArticularyWordRecognition",
    "AtrialFibrillation",
    "BasicMotions",
    "Cricket",
    "DuckDuckGeese",
    "EigenWorms",
    "Epilepsy",
    "EthanolConcentration",
    "ERing",
    "FaceDetection",
    "FingerMovements",
    "HandMovementDirection",
    "Handwriting",
    "Heartbeat",
    "Libras",
    "LSST",
    "MotorImagery",
    "NATOPS",
    "PenDigits",
    "PEMS-SF",
    "PhonemeSpectra",
    "RacketSports",
    "SelfRegulationSCP1",
    "SelfRegulationSCP2",
    "StandWalkJump",
    "UWaveGestureLibrary",
]

In [74]:
fewvariate_equal_length = [
    "AtrialFibrillation",
    "BasicMotions",
    "Cricket",
    "EigenWorms",
    "Epilepsy",
    "EthanolConcentration",
    "ERing",
    "Handwriting",
    "Libras",
    "LSST",
    "PenDigits",
    "RacketSports",
    "SelfRegulationSCP1",
    "SelfRegulationSCP2",
    "StandWalkJump",
    "UWaveGestureLibrary",
]


In [75]:
def subsample(arrays: list[np.ndarray], frac: float, seed: int = 42):
    rng = np.random.default_rng(seed)
    mask: np.ndarray = rng.random(arrays[0].shape[:2]) < frac
    split_indices = np.cumsum(mask.sum(axis=1))[:-1]
    return [np.split(array[mask], split_indices) for array in arrays]

In [76]:
raw_data_dir = os.path.join(os.environ["DATA_DIR"], "raw")
processed_data_dir = os.path.join(os.environ["DATA_DIR"], "preprocessed")


def tsc2pl(tsc_name: str):
    # (N, F, L)
    X_train, y_train = load_classification(
        tsc_name, split="train", extract_path=raw_data_dir
    )
    X_test, y_test = load_classification(
        tsc_name, split="test", extract_path=raw_data_dir
    )

    # Standardize features
    mean = np.mean(X_train, axis=(0, 2), keepdims=True)
    std = np.std(X_train, axis=(0, 2), keepdims=True)

    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    Ntrain, F, L = X_train.shape
    Ntest = X_test.shape[0]

    train_feats = list(X_train.transpose(1, 0, 2))
    test_feats = list(X_test.transpose(1, 0, 2))
    time = np.arange(L)
    time_train = np.repeat(time[None], Ntrain, 0)
    time_test = np.repeat(time[None], Ntest, 0)

    *train_feats, time_train = subsample([*train_feats, time_train], 0.5, seed=42)
    *test_feats, time_test = subsample([*test_feats, time_test], 0.5, seed=42)

    trainval_df = pl.DataFrame(
        {f"x{i}": feat for i, feat in enumerate(train_feats)}
        | {"time": time_train, "target": y_train}
    )

    train_df, val_df = train_test_split(trainval_df, test_size=0.2, random_state=42)

    train_df = train_df.with_columns(split=pl.lit("train"))
    val_df = val_df.with_columns(split=pl.lit("val"))

    test_df = pl.DataFrame(
        {f"x{i}": feat for i, feat in enumerate(test_feats)}
        | {"time": time_test, "target": y_test}
    ).with_columns(split=pl.lit("test"))

    df = pl.concat([train_df, val_df, test_df])
    target_dtype = pl.Boolean if df["target"].n_unique() == 2 else pl.Int32
    df = df.with_columns(
        target=pl.col("target").cast(pl.Categorical).to_physical().cast(target_dtype),
    )

    return df

In [77]:
import re


def to_snake(s):
    # First split sequences of uppercase letters (like "USA")
    s = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1_\2", s)
    # Then insert underscores before uppercase letters
    s = re.sub(r"([a-z])([A-Z])", r"\1_\2", s)
    # Convert to lowercase
    return s.lower()


In [78]:
stats = []

for ds in tqdm(fewvariate_equal_length):
    df = tsc2pl(ds)
    df.write_parquet(os.path.join(processed_data_dir, to_snake(ds) + ".parquet"))
    nfeats = len(df.columns) - 2
    nsamps = len(df)
    seqlen = df["time"].list.len().first()
    nclasses = df["target"].n_unique()
    stats.append(
        {
            "name": ds,
            "nfeats": nfeats,
            "nsamps": nsamps,
            "nclasses": nclasses,
            "seqlen": seqlen,
        }
    )

  0%|          | 0/16 [00:00<?, ?it/s]

In [79]:
with pl.Config(tbl_rows=1000):
    display(
        pl.DataFrame(stats)
        # .filter(
        #     pl.col("nsamps") > 100,
        #     pl.col("seqlen") > 5000,
        #     pl.col("nclasses") < 10,
        # )
    )

name,nfeats,nsamps,nclasses,seqlen
str,i64,i64,i64,i64
"""AtrialFibrillation""",3,30,3,309
"""BasicMotions""",7,80,4,60
"""Cricket""",7,180,12,605
"""EigenWorms""",7,259,5,9066
"""Epilepsy""",4,275,4,112
"""EthanolConcentration""",4,524,4,864
"""ERing""",5,300,6,28
"""Handwriting""",4,1000,26,67
"""Libras""",3,360,15,20
